# 99 - Lessons Learned

A short capstone that steps back from the individual case studies and states what this lab is really about: treating unsupervised learning as a disciplined modelling workflow rather than a parade of algorithms. The recurring theme is **judgement under no supervision** - choosing methods deliberately, validating structure you cannot directly check, and being honest about the limits.

## Lesson 1 - Evaluate without labels, then test stability

Internal metrics (silhouette, Davies-Bouldin, Calinski-Harabasz) describe one fit; they do not tell you whether that fit is *reproducible*. The honest question is whether the structure survives a different seed, a resample, a rescaling, or a few outliers. The cell below pairs a quality metric with a stability metric on the customer data - a single fit looks good *and* the partition is stable across seeds, which is the combination worth trusting.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.cluster import KMeans

sys.path.append(str(Path.cwd() / "src"))

from unsup_lab.data import make_customer_segmentation_data
from unsup_lab.evaluation import evaluate_clustering
from unsup_lab.preprocessing import scale_features
from unsup_lab.stability import (
    pairwise_adjusted_mutual_information,
    repeated_run_labels,
)

x = scale_features(make_customer_segmentation_data(800, random_state=0).features).to_numpy()
factory = lambda seed: KMeans(n_clusters=5, n_init=10, random_state=seed)  # noqa: E731

silhouette = evaluate_clustering(x, factory(0).fit_predict(x)).silhouette
stability = pairwise_adjusted_mutual_information(
    repeated_run_labels(x, factory, n_runs=10, random_state=0)
)

pd.DataFrame(
    [
        {"diagnostic": "silhouette (single fit)", "value": round(silhouette, 3)},
        {"diagnostic": "seed stability (mean AMI)", "value": round(stability.mean_ami, 3)},
    ]
)

ModuleNotFoundError: No module named 'unsup_lab'

## Lesson 2 - One result is not an answer

Reporting a single clustering as *the* segmentation is the most common unsupervised mistake. Bootstrap stability, seed agreement, consensus clustering (notebook 07) and drift monitoring (notebook 08) all exist to answer "how much should I trust this?" before anyone acts on it.

## Lesson 3 - Choose methods by their assumptions

Every method encodes a prior about what a cluster, anomaly or topic *is*. Picking one without naming that assumption hides a modelling decision in plain sight.

| Method | Assumes | Breaks when |
| --- | --- | --- |
| KMeans | compact, spherical clusters | clusters are elongated or unequal density |
| Gaussian Mixture | elliptical, soft clusters | clusters are non-Gaussian |
| DBSCAN | density-connected regions | densities vary a lot across clusters |
| Dirichlet-process GMM | Gaussian components, unknown count | clusters are non-elliptical |
| DTW clustering | shape matters, timing does not | amplitude differences dominate |
| Spectral / modularity | assortative communities | hubs, overlap, degree heterogeneity |

## Lesson 4 - Failure modes are part of the deliverable

A model that only ever shows its best case is not trustworthy. Notebook 04 reproduces topic-model failures (over-factorising, short documents, vocabulary drift, aggressive filtering); the clustering notebooks show where each algorithm struggles. Naming the failure modes is what lets someone rely on the successes.

## Lesson 5 - A notebook is not a product, but it can become one

The modelling logic lives in a typed, tested package, behind a ruff + mypy + pytest + notebook-smoke gate. The same code powers a CLI, versioned artifacts with metadata, JSON reports, an experiment log, a FastAPI service and a Dockerfile - so the distance from research notebook to a running endpoint is short and explicit (see [`docs/workflows.md`](../docs/workflows.md)).

## If I kept going

The honest next steps, not a victory lap:

- Replace synthetic generators with real feature pipelines and data contracts; synthetic data flatters every method.
- Extend drift monitoring from a single feature (PSI) to multivariate or model-based detection.
- Push the deep-clustering work (autoencoder, DEC and contrastive embeddings, now in `unsup_lab.deep`) onto genuinely high-dimensional data such as images, where representation learning pays off most.
- Scale the O(n^2) pieces (consensus co-association, DTW and spectral distance matrices) with sampling or approximate methods.

See [`docs/decision_notes.md`](../docs/decision_notes.md) for the tradeoffs behind the choices made here.